# EcoData · S7 — Ingesta de eventos de sensores en tiempo real (dimensión clima)
**Alahin Reyme Ticona Veliz** · Proyecto Sello EcoData (dimensión **clima / temperatura**, Juliaca) · Big Data, UPeU

Patrón de la guía: `dispositivo → puente → Kafka → consumer con validación de esquema y de rango físico`.

| Guía S7 | EcoData · clima |
|---|---|
| Dispositivo (ESP32) | Estación virtual de Juliaca en Open-Meteo (temperatura a 2 m, humedad, viento) |
| Puente MQTT → Kafka | **`puente-openmeteo-clima`**: llama a la API y publica un evento por hora |
| Topic `atmos-eventos` | **`ecodata-clima-s07-lecturas`** (3 particiones), aparte de `atmos-clima` para no alterar Grafana |
| `consumer_sensores.py` | **`validador-clima`**: separa `invalid` (forma) de `alerta_rango` (valor imposible) y de `alerta_salto` (cambio físicamente imposible en 1 h) |

> ⚠️ **Transparencia sobre la fuente.** Open-Meteo entrega datos **modelados** (reanálisis/modelo de ≈ 10–25 km), no lecturas de un sensor físico propio. El patrón de ingesta es el mismo, pero la rúbrica de S7 pide un sensor real: esto se declara en el informe y la fuente puede reemplazarse por un sensor físico (por ejemplo un ESP32 con DHT22 que mida temperatura y humedad) sin tocar el contrato ni el validador: solo cambia el puente.

Ejecuta las secciones en orden. **No repitas** la sección de publicación (duplica eventos).

## 0. Preparación

In [1]:
import sys, json, time
from datetime import datetime, timezone
import pandas as pd
sys.path.insert(0, "/tmp/s12/scripts")
import clima_comun as clima, openmeteo_clima_a_kafka as pub_clima
from kafka import KafkaProducer, KafkaConsumer, TopicPartition
from kafka.admin import KafkaAdminClient, NewTopic
from kafka.errors import TopicAlreadyExistsError

BOOTSTRAP, TOPIC = "kafka:9092", "ecodata-clima-s07-lecturas"
def log(**kv): print(" ".join(f"{k}={v}" for k, v in kv.items()))

adm = KafkaAdminClient(bootstrap_servers=BOOTSTRAP)
try: adm.create_topics([NewTopic(TOPIC, num_partitions=3, replication_factor=1)]); print("topic creado:", TOPIC)
except TopicAlreadyExistsError: print("el topic ya existía:", TOPIC)
adm.close()

topic creado: ecodata-clima-s07-lecturas


## 1. Contrato de evento propio (4.1, punto 2)
Es el mismo contrato que usa Spark (`clima_comun.ESQUEMA`), ahora validado por un consumidor independiente. **Unidades explícitas**:

| Campo | Tipo | Unidad | Rango físico plausible (alerta si sale) |
|---|---|---|---|
| `tipoEvento` | string | — | `clima.lectura` |
| `sensorId` | string | — | no vacío; es la **key** |
| `temp` | número | °C a 2 m | −30 – 35 (Juliaca está a 3 800 m; el mínimo histórico del proyecto es muy superior a −30) |
| `hum` | número | % humedad relativa | 0 – 100 |
| `viento` | número | km/h a 10 m | 0 – 200 |
| `origen` | string | — | `openmeteo-api`, `openmeteo-replay`, `prueba-falla` |
| `timestamp` | entero | ms, UTC | hora del dato |

**Tres niveles de validación (2.2 de la guía):** (1) *esquema*: campos, tipos, `tipoEvento` → si falla es `invalid`; (2) *rango físico*: la forma es correcta pero el valor es imposible → `alerta_rango`; (3) *coherencia temporal*: la temperatura no puede cambiar más de **15 °C en una hora** respecto de la lectura anterior del mismo sensor → `alerta_salto`.

## 2. Rangos con base en datos reales
Los rangos no se inventan: se contrastan con lo realmente observado para comprobar que los contienen con margen. Se usa la API y el histórico real del proyecto.

In [2]:
try: sc = pub_clima.desde_api()
except Exception as ex: print("API no disponible:", type(ex).__name__); sc = []
hist = clima.leer_json("/tmp/s12/clima_juliaca_2023.json")
def resumen(serie, nombre):
    return {"fuente": nombre, "horas": len(serie),
            "temp_min": min(x[1] for x in serie), "temp_max": max(x[1] for x in serie),
            "hum_min": min(x[2] for x in serie), "hum_max": max(x[2] for x in serie),
            "viento_min": min(x[3] for x in serie), "viento_max": max(x[3] for x in serie),
            "salto_max_1h": max(abs(b[1] - a[1]) for a, b in zip(serie, serie[1:]))}
filas = ([resumen(sc, "API (reciente)")] if sc else []) + [resumen(hist, "histórico 2023")]
pd.DataFrame(filas).round(1)

,fuente,horas,temp_min,temp_max,hum_min,hum_max,viento_min,viento_max,salto_max_1h
0,API (reciente),52,2.0,19.8,19,99,0.6,21.0,5.0
1,histórico 2023,8760,-4.2,21.9,2,100,0.0,39.3,9.4


## 3. Puente `puente-openmeteo-clima` (3.7–3.8)
Convierte cada hora de la API en un evento del contrato y lo publica con `acks=all`. El **sensor no habla Kafka**: el puente es quien traduce (pregunta de defensa 1).

In [3]:
RANGOS = {"temp": (-30, 35), "hum": (0, 100), "viento": (0, 200)}
CAMPOS = {"sensorId": str, "temp": (int, float), "hum": (int, float), "viento": (int, float), "origen": str, "timestamp": int}
SALTO_MAX = 15.0       # °C por hora

prod = KafkaProducer(bootstrap_servers=BOOTSTRAP, acks="all", key_serializer=str.encode,
                     value_serializer=lambda v: v if isinstance(v, bytes) else json.dumps(v).encode())
def enviar(ev, key, verbose=False):
    f = prod.send(TOPIC, key=key, value=ev)
    if verbose:
        m = f.get(timeout=10)
        log(service="puente-openmeteo-clima", component="producer", topic=m.topic, partition=m.partition, offset=m.offset,
            eventType=ev["tipoEvento"] if isinstance(ev, dict) else "RAW", sensorId=key, status="published")

origen = "openmeteo-api" if sc else "openmeteo-replay"
serie_pub = sc if sc else hist[-72:]
ev_clima = [clima.evento(*fila, origen) for fila in serie_pub]
for i, e in enumerate(ev_clima): enviar(e, e["sensorId"], verbose=i < 3)     # ¡UNA sola vez!
prod.flush(); print(f"publicados {len(ev_clima)} eventos de clima (origen {origen}) en {TOPIC}")

service=puente-openmeteo-clima component=producer topic=ecodata-clima-s07-lecturas partition=2 offset=0 eventType=clima.lectura sensorId=openmeteo-clima-juliaca status=published
service=puente-openmeteo-clima component=producer topic=ecodata-clima-s07-lecturas partition=2 offset=1 eventType=clima.lectura sensorId=openmeteo-clima-juliaca status=published
service=puente-openmeteo-clima component=producer topic=ecodata-clima-s07-lecturas partition=2 offset=2 eventType=clima.lectura sensorId=openmeteo-clima-juliaca status=published
publicados 52 eventos de clima (origen openmeteo-api) en ecodata-clima-s07-lecturas


## 4. Fallas provocadas (4.1, evidencia de rango)
La fuente modelada **no produce valores imposibles**, así que, para demostrar que el validador funciona, se inyectan seis lecturas defectuosas, marcadas `origen = prueba-falla` (queda declarado en el informe: son fallas simuladas de un sensor, no datos reales).

In [4]:
t0 = int(time.time() * 1000)
base_c = clima.evento(t0, 5.0, 40.0, 10.0, "prueba-falla")
fallas = {
    "temperatura imposible (-80 °C)":     {**base_c, "temp": -80.0},
    "humedad imposible (140 %)":          {**base_c, "hum": 140.0},
    "viento negativo (-12 km/h)":         {**base_c, "viento": -12.0},
    "salto brusco (+18 °C en 1 h)":       {**base_c, "temp": 23.0, "timestamp": t0 + 3600_000},
    "sin campo hum (esquema)":            {k: v for k, v in base_c.items() if k != "hum"},
}
enviar(base_c, base_c["sensorId"])     # lectura normal previa (referencia para el salto)
for nombre, ev in fallas.items():
    print("→", nombre); enviar(ev, ev["sensorId"], verbose=True)
print("→ texto plano (no es JSON)"); enviar(b"sensor-desconectado", "falla-texto", verbose=True)
prod.flush()

→ temperatura imposible (-80 °C)
service=puente-openmeteo-clima component=producer topic=ecodata-clima-s07-lecturas partition=2 offset=53 eventType=clima.lectura sensorId=openmeteo-clima-juliaca status=published
→ humedad imposible (140 %)
service=puente-openmeteo-clima component=producer topic=ecodata-clima-s07-lecturas partition=2 offset=54 eventType=clima.lectura sensorId=openmeteo-clima-juliaca status=published
→ viento negativo (-12 km/h)
service=puente-openmeteo-clima component=producer topic=ecodata-clima-s07-lecturas partition=2 offset=55 eventType=clima.lectura sensorId=openmeteo-clima-juliaca status=published
→ salto brusco (+18 °C en 1 h)
service=puente-openmeteo-clima component=producer topic=ecodata-clima-s07-lecturas partition=2 offset=56 eventType=clima.lectura sensorId=openmeteo-clima-juliaca status=published
→ sin campo hum (esquema)
service=puente-openmeteo-clima component=producer topic=ecodata-clima-s07-lecturas partition=2 offset=57 eventType=clima.lectura sensorId

## 5. Consumidor `validador-clima` (esquema, rango y coherencia)

In [5]:
def validar(ev, previo=None):
    """Devuelve (status, detalle). Primero esquema, luego rango físico, luego salto entre horas consecutivas."""
    if not isinstance(ev, dict): return "invalid", "no es un objeto JSON"
    if ev.get("tipoEvento") != "clima.lectura": return "invalid", f"tipoEvento desconocido: {ev.get('tipoEvento')}"
    for k, t in CAMPOS.items():
        if k not in ev: return "invalid", f"falta {k}"
        if isinstance(ev[k], bool) or not isinstance(ev[k], t): return "invalid", f"tipo inválido en {k}"
    extra = set(ev) - set(CAMPOS) - {"tipoEvento"}
    if extra: return "invalid", f"campos no permitidos: {sorted(extra)}"
    for campo, (lo, hi) in RANGOS.items():
        if not lo <= ev[campo] <= hi: return "alerta_rango", f"{campo}={ev[campo]} fuera de [{lo}, {hi}]"
    if previo is not None and 0 < ev["timestamp"] - previo["timestamp"] <= 3600_000 and abs(ev["temp"] - previo["temp"]) > SALTO_MAX:
        return "alerta_salto", f"cambio de {ev['temp'] - previo['temp']:+.1f} °C en ≤ 1 h"
    return "ok", ""

def validador(espera_s=8, grupo="validador-clima-group"):
    cons = KafkaConsumer(TOPIC, bootstrap_servers=BOOTSTRAP, group_id=grupo, auto_offset_reset="earliest",
                         consumer_timeout_ms=int(espera_s * 1000))
    filas, previo = [], {}          # previo: última lectura válida por (partición, sensor)
    for m in cons:
        try: ev = json.loads(m.value.decode("utf-8"))
        except Exception: ev = None
        clave = (m.partition, m.key.decode() if m.key else None)
        st, det = validar(ev, previo.get(clave))
        if st in ("ok", "alerta_salto") and ev: previo[clave] = ev
        filas.append((m.partition, m.offset, clave[1], (ev or {}).get("tipoEvento", "—"), st, det))
        if st != "ok":
            log(service="validador-clima", component="consumer", topic=m.topic, partition=m.partition, offset=m.offset,
                status=st, detalle=f'"{det}"', **({"rawPayload": m.value[:40]} if ev is None else {}))
    cons.close()
    return pd.DataFrame(filas, columns=["partición", "offset", "key", "tipoEvento", "status", "detalle"])
df = validador()
print("\nResumen por status:"); print(df.status.value_counts().to_string())

service=validador-clima component=consumer topic=ecodata-clima-s07-lecturas partition=1 offset=0 status=invalid detalle="no es un objeto JSON" rawPayload=b'sensor-desconectado'
service=validador-clima component=consumer topic=ecodata-clima-s07-lecturas partition=2 offset=53 status=alerta_rango detalle="temp=-80.0 fuera de [-30, 35]"
service=validador-clima component=consumer topic=ecodata-clima-s07-lecturas partition=2 offset=54 status=alerta_rango detalle="hum=140.0 fuera de [0, 100]"
service=validador-clima component=consumer topic=ecodata-clima-s07-lecturas partition=2 offset=55 status=alerta_rango detalle="viento=-12.0 fuera de [0, 200]"
service=validador-clima component=consumer topic=ecodata-clima-s07-lecturas partition=2 offset=56 status=alerta_salto detalle="cambio de +18.0 °C en ≤ 1 h"
service=validador-clima component=consumer topic=ecodata-clima-s07-lecturas partition=2 offset=57 status=invalid detalle="falta hum"

Resumen por status:
status
ok              53
alerta_rango  

## 6. Distribución de particiones
Una estación = una *key* = una partición: es lo correcto para conservar el orden de sus lecturas (y para que el chequeo de salto compare horas del mismo sensor). Se muestra la distribución real y debe coincidir con Kafka UI.

In [6]:
dist = df.groupby(["partición", "key"]).size().rename("eventos").reset_index()
display(dist)
print(df.groupby("partición").offset.agg(["min", "max", "count"]))

,partición,key,eventos
0,1,falla-texto,1
1,2,openmeteo-clima-juliaca,58


           min  max  count
partición                 
1            0    0      1
2            0   57     58


## 7. Comparación con la guía (para la reflexión)
| | Guía S7 (ESP32 en Wokwi) | EcoData · clima |
|---|---|---|
| Frecuencia | un evento cada pocos segundos | **1 por hora** (la fuente actualiza cada hora) |
| Fuente | simulada | API real, pero **modelada** |
| Key | `deviceId` | `sensorId` |
| Validación | esquema + rango | esquema + rango + salto máximo de 15 °C/h |
| Qué cambió | — | solo el puente; el contrato y el validador siguen igual |

### Qué capturar (reloj y perfil visibles)
1. Kafka UI → `ecodata-clima-s07-lecturas`: 3 particiones, mensajes y la distribución.
2. Salida de la sección 3 (puente publicando) y de la 4 (fallas publicadas).
3. Salida de la sección 5: logs `status=alerta_rango`, `alerta_salto` e `invalid`, y el resumen por status.
4. Tabla de la sección 2 (rangos vs. valores observados) y la de la sección 6.
5. Si conectas un sensor físico (ESP32 + DHT22), una foto del hardware: es lo que pide la rúbrica de S7.

In [7]:
prod.close()